<a href="https://colab.research.google.com/github/Jaguar838/llm-zoomcamp/blob/main/HW/2026/04-evaluation/hw-04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Homework: Evaluation

In homework 2 we built keyword, vector, and hybrid search over the course
lessons, and ended with an open question: which one is best? The way to answer
that is to measure, and that's what we do here.

In this homework we generate a ground truth dataset and use it to evaluate
search, the same way we did in the module. There we only evaluated keyword
search. Here we also evaluate vector and hybrid search, so we can finally
compare them on numbers instead of intuition.

Like in homework 1 and 2, our knowledge base is the course lessons themselves.
Each module has a `lessons/` folder of numbered markdown pages, and we pull
them from GitHub. We use commit `8c1834d`, so everyone works with the exact
same 72 pages.

> It's possible your answers won't match exactly. If so, select the closest one.

## Setup

This homework continues from homework 2. We reuse the same chunks and the same
search functions, so it's easiest to keep working in the same project.

We need a few more libraries for generating questions with an LLM:

In [ ]:
!pip install pydantic gitsource groq sqlitesearch onnxruntime minsearch

For the LLM, we recommend OpenAI with `gpt-5.4-mini`, but you can use any model
and provider you like - just adapt the client accordingly. Put your key in a
`.env` file as in the earlier modules.

Load the data exactly as in homework 2:

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
# Convert RawRepositoryFile objects to dictionaries suitable for chunk_documents
documents = [{
    'filename': file.filename,
    'content': file.content
} for file in reader.read()]

In [3]:
len(documents)

72

This gives 72 pages.

## Generating ground truth

To evaluate search, we need a dataset of questions where we know which document
is the correct answer. This is the ground truth.

We generate it the same way as in the module. For each lesson page, we ask an
LLM to write 5 questions that are answered by that page. Each question is then
labeled with the page it came from.

We use the same structured-output approach as in the module - the same
`Questions` model and the `llm_structured` helper from `evaluation_utils.py`.

Download `evaluation_utils.py` and the `rag_helper.py` it depends on:


```
%%bash
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
wget "${PREFIX}/01-agentic-rag/code/rag_helper.py"
wget "${PREFIX}/04-evaluation/code/evaluation_utils.py"
```

The module's instructions generate questions from a FAQ record, so we adapt
them for a lesson page:

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
- Your output should be a JSON object.
""".strip()

We ask for different wording from the page on purpose. Real users don't phrase
their questions the way the lesson does, and copying the text would make the
evaluation too easy.

For each page, build a JSON user prompt from its `filename` and `content`, then
call `llm_structured` with the `Questions` model. Turn each returned question
into a record labeled with the page's `filename`. The call also returns the
token usage, the same as in the lessons.

## Q1. Generating questions

Generating questions for all 72 pages costs money and takes time, so let's
start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`

Each call returns the token usage, which most LLM APIs report on the response
object (e.g. `input_tokens` / `prompt_tokens`).

What's the average number of input tokens across these 3 calls?

* 140
* `1400`
* 14000
* 140000

> These numbers vary between runs, even with the same model, so pick the closest
> option. A different provider or model may land further apart, but the input
> tokens stay in the same order of magnitude - the prompt we send is the same.


In [5]:
from groq import Groq
from google.colab import userdata

# Retrieve Groq API Key from Colab secrets
# Please ensure you have a 'GROQ_API_KEY' secret configured in Colab.
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab secrets. Please add it to proceed.")

# Initialize Groq client
groq_client = Groq(api_key=GROQ_API_KEY)

# Define the target filenames as specified in the problem description
target_filenames = [
    '01-agentic-rag/lessons/01-intro.md',
    '01-agentic-rag/lessons/02-environment.md',
    '01-agentic-rag/lessons/03-rag.md',
]

In [6]:
def llm_structured(client, instructions, user_prompt, output_type, model="openai/gpt-oss-20b"):
    # Groq's chat.completions.create expects system and user roles
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": json.dumps(user_prompt)} # user_prompt is a dict, dump it to JSON string
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        response_format={"type": "json_object"} # Specify JSON object output
    )

    # The content will be a JSON string, parse it and validate with Pydantic model
    json_output = json.loads(response.choices[0].message.content)
    parsed_output = output_type(**json_output) # Validate and parse with Pydantic model

    return parsed_output, response.usage

def llm_structured_retry(
    client,
    instructions,
    user_prompt,
    output_type,
    model="openai/gpt-oss-20b",
    max_retries=3,
):
    for attempt in range(max_retries):
        try:
            return llm_structured(
                client,
                instructions,
                user_prompt,
                output_type,
                model=model,
            )
        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)
# --- End of redefinition ---

In [9]:
from pydantic import BaseModel
import time
import json

class Questions(BaseModel):
    questions: list[str]

# Filter documents to get only the target lessons
target_documents = [d for d in documents if any(f in d['filename'] for f in target_filenames)]

input_tokens_list = []

for doc in target_documents:
    # Prepare the user prompt for question generation
    user_prompt = {
        'filename': doc['filename'],
        'content': doc['content']
    }

    # Generate questions and capture token usage
    _, usage = llm_structured_retry(
        client=groq_client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
        # model="llama-3.3-70b-versatile"
    )
    input_tokens_list.append(usage.prompt_tokens)

# Calculate the average input tokens
average_input_tokens = sum(input_tokens_list) / len(input_tokens_list)

print(f"Input tokens for each call: {input_tokens_list}")
print(f"Average input tokens across the 3 calls: {average_input_tokens:.0f}")

Input tokens for each call: [1086, 1352, 1819]
Average input tokens across the 3 calls: 1419


## The full ground truth

You don't need to generate the data for the rest of the homework. We already
did it for all 72 pages, using the same approach as in the lessons, and saved
the 360 questions to a file.

Download it:

In [10]:
%%bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

--2026-07-14 00:13:24--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv’

     0K .......... .......... .......... .......... .......   100% 4.67M=0.01s

2026-07-14 00:13:25 (4.67 MB/s) - ‘ground-truth.csv’ saved [48627/48627]



Load it with pandas into a list of records called `ground_truth`. Each record
has a `question` and the `filename` of the page that should answer it.


In [11]:
import pandas as pd

ground_truth = pd.read_csv('ground-truth.csv')
display(ground_truth.head())

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


## Searching the chunks

We search over the same chunks as in homework 2.

Create them with `chunk_documents`:

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
len(chunks)

295

This gives 295 chunks.

Now rebuild the search from homework 2 over these chunks. Build a text index
(`Index`) and a vector index (`VectorSearch`), both keyed on `filename`. Wrap
each one in a function, `text_search` and `vector_search`, that takes a query
and the number of results to return (5 by default).

For hybrid search, reuse the `rrf` function from homework 2:

In [14]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Then define `hybrid_search` on top of it:

In [15]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search

Take the first question from the ground truth:

In [16]:
# Accessing the question from the first row using .iloc
q = ground_truth.iloc[0]["question"]
print(f"Question: {q}")

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


After running `text_search` for it, what's the `filename` of the first result?

* 01-agentic-rag/lessons/01-intro.md
* `01-agentic-rag/lessons/03-rag.md`
* 01-agentic-rag/lessons/13-function-calling.md
* 01-agentic-rag/lessons/10-rag-next-steps.md

In [30]:
%%time
# Q2: First result with text search
# Re-initializing text_index based on 'chunks' as per the assignment instructions
from sqlitesearch import TextSearchIndex # Changed from SQLiteSearch

# The assignment specifies the index should be built over chunks
# TextSearchIndex expects text_fields and keyword_fields to be specified during initialization
text_index = TextSearchIndex(
    text_fields=['content'],
    keyword_fields=['filename'],
    db_path="lesson_llm_course_chunks.db" # Use a dedicated database file for the text index
)

# Add each chunk to the index
# The 'chunks' list already contains dictionaries with 'content' and 'filename'
for chunk in chunks:
    text_index.add(chunk)

def text_search(query, num_results=5):
    query_text = query if isinstance(query, str) else query.get('question', '')
    # TextSearchIndex's search method can take a limit for num_results
    results = text_index.search(query_text, num_results=5)
    return results

# Get the first question
q = ground_truth.iloc[0]["question"]

# Perform text search
results = text_search(q)

if results:
    print(f"First result filename: {results[0]['filename']}")
else:
    print("No results found.")

First result filename: 01-agentic-rag/lessons/03-rag.md
CPU times: user 92.1 ms, sys: 25.8 ms, total: 118 ms
Wall time: 216 ms


### Demonstrating a Search with `sqlitesearch`

Once the index is built, you can perform searches using the `search` method, specifying your query and the number of top results you want (`top_n`).

In [34]:
search_query = "What is RAG?"
results = text_index.search(search_query, num_results=3)

print(f"\nSearch results for: '{search_query}'")
for i, res in enumerate(results):
    print(f"Result {i+1}:")
    print(f"  Filename: {res['filename']}")
    print(f"  Content snippet: {res['content'][:150]}...") # Print a snippet of the content



Search results for: 'What is RAG?'
Result 1:
  Filename: 02-vector-search/lessons/06-rag-vector.md
  Content snippet: # RAG with Vector Search

Video: [Watch this lesson](https://www.youtube.com/watch?v=-GBW3g3PVTM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In module 1...
Result 2:
  Filename: 02-vector-search/lessons/06-rag-vector.md
  Content snippet: # RAG with Vector Search

Video: [Watch this lesson](https://www.youtube.com/watch?v=-GBW3g3PVTM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In module 1...
Result 3:
  Filename: 03-orchestration/lessons/05-rag.md
  Content snippet: be accurate.

### Step 2: With RAG

Flow: [`2_chat_with_rag.yaml`](../flows/2_chat_with_rag.yaml)

This flow:

1. Ingests the Kestra 1.1 release blog ...


## Q3. First result with vector search

After running `vector_search` for the same question, what's the `filename` of
the first result?

* `01-agentic-rag/lessons/01-intro.md`
* 01-agentic-rag/lessons/03-rag.md
* 04-evaluation/lessons/11-evaluation-intro.md
* 04-evaluation/lessons/12-rag-answers.md

This question was generated from `01-agentic-rag/lessons/01-intro.md`. Notice
that one method finds the right page at the top and the other doesn't. That's
exactly why we measure across the whole dataset instead of trusting one query.

In [18]:
%%bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget -q $PREFIX/download.py -O download.py
wget -q $PREFIX/embedder.py -O embedder.py
python download.py

  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


In [42]:
%%time
from embedder import Embedder
from sqlitesearch import VectorSearchIndex
import numpy as np
from tqdm.auto import tqdm

# 1. Load the alternative model
model_path = 'models/Xenova/all-MiniLM-L6-v2'
embedder = Embedder(model_path)

# 2. Prepare vectors
chunk_vectors = []
for chunk in tqdm(chunks, desc="Indexing chunks (all-MiniLM-L6-v2)"):
    v = embedder.encode(chunk['content'])
    chunk_vectors.append(v)

chunk_vectors = np.array(chunk_vectors)

# 3. Fit VectorSearch index
vindex = VectorSearchIndex(keyword_fields=['filename'])
vindex.fit(chunk_vectors, chunks)

# 4. Search
def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vindex.search(query_vector, num_results=num_results)

# 5. Q3 calculation
q = ground_truth.iloc[0]["question"]
vector_results_alt = vector_search(q)

if vector_results_alt:
    print(f"Q3 Result for all-MiniLM-L6-v2: {vector_results_alt[0]['filename']}")

Indexing chunks (all-MiniLM-L6-v2):   0%|          | 0/295 [00:00<?, ?it/s]

Q3 Result for all-MiniLM-L6-v2: 01-agentic-rag/lessons/01-intro.md
CPU times: user 20.5 s, sys: 334 ms, total: 20.8 s
Wall time: 21.7 s


## Evaluation metrics

We evaluate search exactly as in the module, reusing the same functions from the
lecture. We change only the label. Our ground truth uses `filename`, so a result
counts as a hit when a returned chunk's `filename` matches the question's
`filename`, not a document `id`.

As a reminder, these functions do the following:

- `compute_relevance` runs search for a question and returns a list of 0s and 1s
- `hit_rate` is the fraction of questions where the correct page appears in the results
- `mrr` (Mean Reciprocal Rank) also rewards finding the page near the top
- `evaluate` runs a search function over the whole ground truth and returns both metrics

## Q4. Evaluating text search

Evaluate `text_search` on the ground truth data.

What's the Hit Rate?

* 0.55
* 0.66
* `0.76`
* 0.88

In [36]:
def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if True in line:
            cnt = cnt + 1
    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)
                break
    return total_score / len(relevance_total)

In [37]:
%%time
from tqdm.auto import tqdm

def compute_relevance(ground_truth, search_function):
    relevance_total = []
    for i in tqdm(range(len(ground_truth))):
        rec = ground_truth.iloc[i]
        expected_filename = rec['filename']
        results = search_function(rec['question'])
        relevance = [d['filename'] == expected_filename for d in results]
        relevance_total.append(relevance)
    return relevance_total

# Q4. Evaluating text search
relevance_text = compute_relevance(ground_truth, text_search)
print(f"Text Search Hit Rate: {hit_rate(relevance_text):.2f}")
print(f"Text Search MRR: {mrr(relevance_text):.2f}")

  0%|          | 0/360 [00:00<?, ?it/s]

Text Search Hit Rate: 0.71
Text Search MRR: 0.59
CPU times: user 989 ms, sys: 7.98 ms, total: 997 ms
Wall time: 993 ms


## Q5. Evaluating vector search

Now evaluate `vector_search` - the part we left for the homework, since the
module only evaluated keyword search.

What's the MRR?

* 0.35
* 0.45
* `0.55`
* 0.65

In [43]:
# Q5. Evaluating vector search
relevance_vector = compute_relevance(ground_truth, vector_search)
print(f"Vector Search Hit Rate: {hit_rate(relevance_vector):.3f}")
print(f"Vector Search MRR: {mrr(relevance_vector):.3f}")

  0%|          | 0/360 [00:00<?, ?it/s]

Vector Search Hit Rate: 0.725
Vector Search MRR: 0.549


## Q6. Tuning hybrid search

The `k` constant in RRF controls how much the top ranks matter. A smaller `k`
sharpens the gap between positions, so being at the top of a list counts for
more. The RRF paper uses 60 as a default, but the best value depends on the data
- so let's measure it.

Evaluate `hybrid_search` over the full ground truth dataset for `k` values 1,
50, 100, and 200. Compare the MRR values for these runs.

Which `k` gives the best MRR?

* 1
* 50
* 100
* 200

> Several values of `k` may give the same MRR. If there's a tie, pick the
> smallest `k`.

In [44]:
# Q6. Tuning hybrid search
results_q6 = {}
for k_val in [1, 50, 100, 200]:
    def current_hybrid_search(q, num_results=5):
        return hybrid_search(q, k=k_val)
    relevance_hybrid = compute_relevance(ground_truth, current_hybrid_search)
    current_mrr = mrr(relevance_hybrid)
    results_q6[k_val] = current_mrr
    print(f"Hybrid Search (k={k_val}) MRR: {current_mrr:.4f}")

  0%|          | 0/360 [00:00<?, ?it/s]

Hybrid Search (k=1) MRR: 0.6585


  0%|          | 0/360 [00:00<?, ?it/s]

Hybrid Search (k=50) MRR: 0.6702


  0%|          | 0/360 [00:00<?, ?it/s]

Hybrid Search (k=100) MRR: 0.6702


  0%|          | 0/360 [00:00<?, ?it/s]

Hybrid Search (k=200) MRR: 0.6702


In [46]:
# Find the k with the best MRR, prioritizing smaller k in case of a tie
best_k = -1
max_mrr = -1.0

for k_val in sorted(results_q6.keys()): # Sort to ensure smallest k is picked first for ties
    if results_q6[k_val] > max_mrr:
        max_mrr = results_q6[k_val]
        best_k = k_val
    elif results_q6[k_val] == max_mrr: # If MRR is equal, keep the smaller k (due to sorting)
        best_k = min(best_k, k_val)

print(f"\nBest k value for Hybrid Search based on MRR: {best_k} with MRR: {max_mrr:.4f}")


Best k value for Hybrid Search based on MRR: 50 with MRR: 0.6702


## Using this framework

You now have an `evaluate` function that takes any search function and returns
Hit Rate and MRR.

Use it to measure any change you make to search:

- tune the field boosts in keyword search
- try a different embedding model for vector search
- change `k` in the RRF formula for hybrid search
- change the number of results you return

Change a setting, re-run `evaluate`, and see whether the metric moves. The
ground truth stays fixed, so the comparison is fair. That's how you replace
guessing with measuring.

In [47]:
%%writefile rag_helper_groq.py
from dataclasses import dataclass
from rag_helper import RAGBase as OriginalRAGBase

# Define a dataclass to store the result of a RAG query, including the answer and token counts.
@dataclass
class RAGResult:
    answer: str
    input_tokens: int
    output_tokens: int

# Define the RAGBase class by inheriting from OriginalRAGBase and adapting it for our schema.
class RAGBase(OriginalRAGBase):

    def __init__(self, index, llm_client, instructions, prompt_template, model="openai/gpt-oss-20b"):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    # Method to perform a search using the provided index.
    # Adapted for the filename/content schema.
    def search(self, query, num_results=1):
        # Assuming index expects a string query, extract from dict if needed
        search_query = query if isinstance(query, str) else query.get('question', '')
        return self.index.search(search_query, num_results=num_results)

    # Method to build context string from search results.
    # Adapted for the filename/content schema.
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    def build_prompt(self, question, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(question=question, context=context).strip()

    # Method to interact with the LLM.
    # Overridden to use chat.completions.create and return the full response object for token counting.
    def llm(self, prompt):
        input_messages = [
            {"role": "system", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=input_messages
        )

        return response

    # Main RAG method to perform the entire RAG process.
    # Overridden to return a RAGResult with token counts.
    def rag(self, query):
        search_results = self.search(query)
        # print(search_results)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)


        # Extract the answer and token usage from the LLM response.
        answer = response.choices[0].message.content
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens

        return RAGResult(answer=answer, input_tokens=input_tokens, output_tokens=output_tokens)

Writing rag_helper_groq.py


In [48]:
%%writefile evaluation_utils.py
import time
import json
from tqdm.auto import tqdm


def calc_price(usage):
    input_price_per_million = 0.75
    output_price_per_million = 4.50

    # Corrected attribute names for Groq CompletionUsage object
    input_cost = (usage.prompt_tokens / 1_000_000) * input_price_per_million
    output_cost = (usage.completion_tokens / 1_000_000) * output_price_per_million
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


def calc_total_price(usages):
    total_cost = 0.0

    for usage in usages:
        cost = calc_price(usage)
        total_cost = total_cost + cost["total_cost"]

    return total_cost


def llm_structured(client, instructions, user_prompt, output_type, model="openai/gpt-oss-20b"):
    # Groq's chat.completions.create expects system and user roles
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": json.dumps(user_prompt)} # user_prompt is a dict, dump it to JSON string
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        response_format={"type": "json_object"} # Specify JSON object output
    )

    # The content will be a JSON string, parse it and validate with Pydantic model
    json_output = json.loads(response.choices[0].message.content)
    parsed_output = output_type(**json_output) # Validate and parse with Pydantic model

    return parsed_output, response.usage

def llm_structured_retry(
    client,
    instructions,
    user_prompt,
    output_type,
    model="openai/gpt-oss-20b",
    max_retries=3,
):
    for attempt in range(max_retries):
        try:
            return llm_structured(
                client,
                instructions,
                user_prompt,
                output_type,
                model=model,
            )
        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)
# --- End of redefinition ---


class RAGWithUsage(RAGBase):

    def __init__(self, index, llm_client, instructions, prompt_template, model="openai/gpt-oss-20b"):
        super().__init__(index, llm_client, instructions, prompt_template, model)
        self.usages = []
        self.last_usage = None

    def reset_usage(self):
        self.usages = []
        self.last_usage = None

    def search(self, query, num_results=5):
        # Extract the actual query string from the input 'query' dict
        search_query = query if isinstance(query, str) else query.get('question', '')
        # Simplified search to match RAGBase.search and the current index structure
        # Assuming text_index is built on 'content' and 'filename' fields
        return self.index.search(search_query, num_results=num_results)

    def llm(self, prompt):
        input_messages = [
            {"role": "system", "content": self.instructions}, # Use system role for instructions
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.chat.completions.create( # Use chat.completions.create
            model=self.model,
            messages=input_messages
        )

        self.last_usage = response.usage
        self.usages.append(response.usage)

        # Return the full response object for compatibility with RAGBase.rag
        return response

    def total_cost(self):
        return calc_total_price(self.usages)


def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results

Writing evaluation_utils.py
